# External Lab Evaluation V2 Corrected

This notebook builds on the corrected task:

```text
query:  metabolite_name
target: refmet_name, or any retrieved name with the same mapped PubChem CID
```

V2 adds the recommended next steps:

1. improve deterministic `refmet_name -> CID` mapping coverage,
2. treat SapBERT as a top-k candidate generator,
3. rerank IVFPQ candidates with exact embedding dot products,
4. add lexical reranking features over the candidate pool,
5. optionally add mapped RefMet labels as a small target dictionary without rebuilding the 114M-name index.

The original corrected notebook is left unchanged.


## Install Dependencies


In [ ]:
import subprocess
import sys

base_packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'duckdb>=1.0',
    'tqdm>=4.66',
    'PyYAML>=6.0',
    'torch>=2.1',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
    'matplotlib>=3.8',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *base_packages])

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu-cu12'])
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu>=1.8'])

print('Installed notebook dependencies.')


## Mount Drive and Locate Repo


In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import shutil
import sys
import time
import unicodedata
from difflib import SequenceMatcher

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer.')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration


In [ ]:
from chem_synonymizer.embed import load_hf_model, pool_outputs
from chem_synonymizer.index import build_faiss_index_from_memmap, faiss_gpu_available
from chem_synonymizer.utils import save_json

# Inputs.
LAB_CSV_PATH = REPO_DIR / 'data' / 'allmetnames_062626.csv'
REFERENCE_CSV_PATH = REPO_DIR / 'data' / 'CID-Synonym-full.csv'
MODEL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'

METABOLITE_COL = 'metabolite_name'
REFMET_COL = 'refmet_name'
CID_COL = 'cid'
NAME_COL = 'name'

RUN_NAME = 'sapbert_contrastive_refmet_v2_corrected'
RESULTS_RUN_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_v2_corrected'
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'

# Reuse existing full-corpus artifacts when available.
BASELINE_CORRECTED_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_refmet_corrected'
OLD_EXTERNAL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_unique_surface'
SHARED_ARTIFACT_DIRS = [
    BASELINE_CORRECTED_DIR / 'artifacts',
    OLD_EXTERNAL_DIR / 'artifacts',
]

SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').exists() else Path('/content')
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / RUN_NAME
WORK_ARTIFACTS_DIR = WORK_DIR / 'artifacts'
PREPARE_DB_PATH = WORK_ARTIFACTS_DIR / 'unique_surface_v2_corrected.duckdb'
CORPUS_PARQUET_PATH = WORK_ARTIFACTS_DIR / 'unique_surface_corpus.parquet'
CORPUS_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'corpus_embeddings.npy'
QUERY_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'metabolite_query_embeddings.npy'
FAISS_INDEX_PATH = WORK_ARTIFACTS_DIR / 'faiss_ivfpq.index'
NEIGHBOR_INDICES_PATH = WORK_ARTIFACTS_DIR / 'metabolite_neighbor_indices.npy'
NEIGHBOR_SCORES_PATH = WORK_ARTIFACTS_DIR / 'metabolite_neighbor_scores.npy'
REFMET_TARGET_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'refmet_target_embeddings.npy'
REFMET_TARGET_INDICES_PATH = WORK_ARTIFACTS_DIR / 'refmet_target_neighbor_indices.npy'
REFMET_TARGET_SCORES_PATH = WORK_ARTIFACTS_DIR / 'refmet_target_neighbor_scores.npy'

# Model/index settings matched to the earlier A100 run.
SEED = 0
MAX_LENGTH = 96
EMBED_DTYPE = 'float16'
POOLING = 'cls'
L2_NORMALIZE = True
CORPUS_ENCODE_BATCH_SIZE = 8192
QUERY_ENCODE_BATCH_SIZE = 8192

INDEX_TYPE = 'ivfpq'
NLIST = 16384
NPROBE = 512
TRAIN_SIZE = 3_000_000
ADD_BATCH_SIZE = 250_000
SEARCH_BATCH_SIZE = 8192
K_RETRIEVE = 256
M_PQ = 96
NBITS = 8
USE_FLOAT16_FAISS = True
USE_GPU_FAISS_IF_FITS = True
GPU_INDEX_MEMORY_LIMIT_GB = 74.0

# V2 settings.
RUN_COMPACT_REFMET_MAPPING = True
MAX_COMPACT_MATCHES_PER_REFMET = 25
MIN_COMPACT_KEY_LEN = 6
RERANK_POOL_K = 256
SAVE_TOP_K = 10
EVALUATE_REFMET_TARGET_DICTIONARY = True
REFMET_TARGET_K = 32

# Strict main metrics use deterministic mapping only. Embedding-assisted label mapping is intentionally
# not used by default because it can make labels depend on the same model being evaluated.
RUN_EMBEDDING_ASSISTED_REFMET_MAPPING = False
USE_EMBEDDING_ASSISTED_MAPPING_IN_MAIN_METRICS = False
EMBED_MAP_MIN_SCORE = 0.90
EMBED_MAP_MIN_MARGIN = 0.04
EMBED_MAP_MIN_LEXICAL_RATIO = 0.80

REBUILD_PREPARED_CORPUS = False
REENCODE_CORPUS = False
REENCODE_QUERIES = False
REBUILD_FAISS_INDEX = False
RERUN_MAIN_SEARCH = False
REENCODE_REFMET_TARGETS = False
RERUN_REFMET_TARGET_SEARCH = False

BACKUP_LARGE_ARTIFACTS_TO_RESULTS = False

for required in [LAB_CSV_PATH, REFERENCE_CSV_PATH, MODEL_DIR]:
    if not required.exists():
        raise FileNotFoundError(required)
for path in [RESULTS_RUN_DIR, RESULTS_ARTIFACTS_DIR, WORK_ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Lab CSV:', LAB_CSV_PATH)
print('Reference CSV:', REFERENCE_CSV_PATH)
print('Model:', MODEL_DIR)
print('Results:', RESULTS_RUN_DIR)
print('Scratch:', WORK_DIR, shutil.disk_usage(SCRATCH_ROOT))


## Normalization, Lexical Features, and Artifact Helpers


In [ ]:
GREEK_MAP = {
    'α': 'alpha', 'β': 'beta', 'γ': 'gamma', 'δ': 'delta',
    'ε': 'epsilon', 'κ': 'kappa', 'λ': 'lambda', 'μ': 'mu',
    'ω': 'omega',
}


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKC', str(value))
    for src, dst in GREEK_MAP.items():
        text = text.replace(src, dst)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def normalize_series(values: pd.Series) -> pd.Series:
    return values.fillna('').astype('string').map(normalize_text).astype('string')


def clean_instrument_suffix(text: str) -> str:
    s = normalize_text(text)
    s = re.sub(r'\[[^\]]*m[^\]]*\]', ' ', s)
    s = re.sub(r'[@_]\s*[-+]?\d+(?:\.\d+)?\s*$', ' ', s)
    s = re.sub(r'\s+[+-]\s*\d+(?:\.\d+)?\s*$', ' ', s)
    s = s.replace('*', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def punctuation_spaced(text: str) -> str:
    s = clean_instrument_suffix(text)
    s = re.sub(r'[\(\)\[\]\{\};,/_]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def compact_key(text: object) -> str:
    return re.sub(r'[^a-z0-9]+', '', normalize_text(text))


def make_refmet_variants(text: object) -> list[tuple[str, str]]:
    base = normalize_text(text)
    variants = [('exact', base)]
    cleaned = clean_instrument_suffix(base)
    spaced = punctuation_spaced(base)
    no_quotes = re.sub(r'[\'"`]', '', cleaned)
    no_terminal_count = re.sub(r'(?:\s|_)+(?:\d+)$', '', cleaned).strip()
    for method, value in [
        ('strip_instrument_suffix', cleaned),
        ('punctuation_spaced', spaced),
        ('remove_quotes', no_quotes),
        ('remove_terminal_count', no_terminal_count),
    ]:
        value = normalize_text(value)
        if value and value not in [v for _, v in variants]:
            variants.append((method, value))
    return variants


def pipe_split(value: object) -> set[str]:
    if pd.isna(value):
        return set()
    return {part.strip() for part in str(value).split('|') if part.strip()}


def pipe_join(values) -> str:
    out = []
    seen = set()
    for value in values:
        for part in pipe_split(value):
            if part not in seen:
                out.append(part)
                seen.add(part)
    return '|'.join(out)


def has_cid_overlap(candidate_cids: object, gold_cids: object) -> bool:
    return bool(pipe_split(candidate_cids) & pipe_split(gold_cids))


def q(path: Path) -> str:
    return str(path).replace("'", "''")


def char_ngrams(text: str, n: int = 3) -> set[str]:
    s = compact_key(text)
    if not s:
        return set()
    if len(s) <= n:
        return {s}
    return {s[i:i+n] for i in range(len(s) - n + 1)}


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))


def token_set(text: str) -> set[str]:
    return {tok for tok in re.split(r'[^a-z0-9]+', normalize_text(text)) if tok}


def sequence_ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def lipid_signature(text: str) -> tuple[str, str, str] | None:
    s = normalize_text(text)
    m = re.search(r'\b([a-z]{1,6})\s*[\( ]?\s*(\d{1,2})\s*:\s*(\d{1,2})\b', s)
    if not m:
        return None
    return (m.group(1), m.group(2), m.group(3))


def stereo_signature(text: str) -> tuple[str, ...]:
    s = normalize_text(text)
    toks = re.findall(r'\b(?:cis|trans|alpha|beta)\b|\b[dl]-|\([+-]\)|\b[0-9]+[rs]\b|\b[0-9]+[ez]\b', s)
    return tuple(sorted(toks))


def lexical_features(query_text: str, candidate_text: str) -> dict:
    qn = normalize_text(query_text)
    cn = normalize_text(candidate_text)
    q_lipid = lipid_signature(qn)
    c_lipid = lipid_signature(cn)
    q_stereo = stereo_signature(qn)
    c_stereo = stereo_signature(cn)
    return {
        'query_exact_candidate': float(qn == cn),
        'char3_jaccard': float(jaccard(char_ngrams(qn, 3), char_ngrams(cn, 3))),
        'token_jaccard': float(jaccard(token_set(qn), token_set(cn))),
        'sequence_ratio': float(sequence_ratio(qn, cn)),
        'lipid_signature_match': float(q_lipid is not None and q_lipid == c_lipid),
        'lipid_class_match': float(q_lipid is not None and c_lipid is not None and q_lipid[0] == c_lipid[0]),
        'stereo_mismatch': float(bool(q_stereo and c_stereo and q_stereo != c_stereo)),
    }


def artifact_candidates(work_path: Path) -> list[Path]:
    return [RESULTS_ARTIFACTS_DIR / work_path.name] + [d / work_path.name for d in SHARED_ARTIFACT_DIRS]


def parts_dir_candidates(work_path: Path) -> list[Path]:
    return [p.with_suffix(p.suffix + '.parts') for p in artifact_candidates(work_path)]


def restore_artifact(work_path: Path) -> bool:
    if work_path.exists():
        return True
    for candidate in artifact_candidates(work_path):
        if candidate.exists():
            work_path.parent.mkdir(parents=True, exist_ok=True)
            print('Restoring artifact:', candidate, '->', work_path)
            shutil.copy2(candidate, work_path)
            for suffix in ['.meta.json', '.timing.json']:
                sidecar = candidate.with_suffix(candidate.suffix + suffix)
                if sidecar.exists():
                    shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))
            return True
    for parts_dir in parts_dir_candidates(work_path):
        if parts_dir.exists():
            parts = sorted(parts_dir.glob('part-*'))
            if not parts:
                continue
            work_path.parent.mkdir(parents=True, exist_ok=True)
            print(f'Reconstructing {work_path.name} from {len(parts)} parts in {parts_dir}...')
            with work_path.open('wb') as out:
                for part in tqdm(parts, desc=f'RESTORE {work_path.name}'):
                    with part.open('rb') as src:
                        shutil.copyfileobj(src, out, length=64 * 1024 * 1024)
            for suffix in ['.meta.json', '.timing.json']:
                sidecar = parts_dir / f'{work_path.name}{suffix}'
                if sidecar.exists():
                    shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))
            return True
    return False


def backup_artifact(work_path: Path, large: bool = False) -> None:
    if large and not BACKUP_LARGE_ARTIFACTS_TO_RESULTS:
        return
    if not work_path.exists():
        return
    target = RESULTS_ARTIFACTS_DIR / work_path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size == work_path.stat().st_size:
        return
    print('Backing up artifact:', work_path, '->', target)
    shutil.copy2(work_path, target)
    for suffix in ['.meta.json', '.timing.json']:
        sidecar = work_path.with_suffix(work_path.suffix + suffix)
        if sidecar.exists():
            shutil.copy2(sidecar, target.with_suffix(target.suffix + suffix))


def load_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


## Save Run Config


In [ ]:
def file_fingerprint(path: Path) -> dict:
    return {
        'path': str(path),
        'exists': path.exists(),
        'size': int(path.stat().st_size) if path.exists() else None,
        'mtime': float(path.stat().st_mtime) if path.exists() else None,
    }


save_json(
    {
        'run_name': RUN_NAME,
        'task': 'v2 corrected metabolite_name to refmet_name / CID evaluation',
        'lab_csv': file_fingerprint(LAB_CSV_PATH),
        'reference_csv': file_fingerprint(REFERENCE_CSV_PATH),
        'model_dir': str(MODEL_DIR),
        'max_length': MAX_LENGTH,
        'pooling': POOLING,
        'embedding_dtype': EMBED_DTYPE,
        'index_type': INDEX_TYPE,
        'nprobe': NPROBE,
        'k_retrieve': K_RETRIEVE,
        'rerank_pool_k': RERANK_POOL_K,
        'v2_steps': [
            'deterministic refmet mapping variants',
            'compact-key refmet mapping',
            'exact embedding rerank over IVFPQ candidates',
            'hybrid lexical rerank',
            'optional refmet target dictionary augmentation',
        ],
    },
    RESULTS_RUN_DIR / 'config.json',
)


## Build or Restore the Unique Reference Surface Corpus


In [ ]:
def prepare_unique_surface_corpus() -> dict:
    restore_artifact(CORPUS_PARQUET_PATH)
    if CORPUS_PARQUET_PATH.exists() and not REBUILD_PREPARED_CORPUS:
        pf = pq.ParquetFile(CORPUS_PARQUET_PATH)
        summary = {
            'loaded_from_cache': True,
            'unique_surface_rows': int(pf.metadata.num_rows),
            'corpus_parquet_path': str(CORPUS_PARQUET_PATH),
        }
        save_json(summary, RESULTS_RUN_DIR / 'corpus_summary.json')
        print(json.dumps(summary, indent=2))
        return summary

    for path in [PREPARE_DB_PATH, CORPUS_PARQUET_PATH]:
        if path.exists():
            path.unlink()

    con = duckdb.connect(str(PREPARE_DB_PATH))
    con.execute(f'PRAGMA threads={max(1, os.cpu_count() or 8)}')
    con.execute('PRAGMA preserve_insertion_order=false')
    con.execute('DROP TABLE IF EXISTS reference_pairs')
    con.execute('CREATE TABLE reference_pairs(normalized_name VARCHAR, cid VARCHAR, display_name VARCHAR)')

    total_rows = 0
    inserted_rows = 0
    t0 = time.perf_counter()
    reader = pd.read_csv(
        REFERENCE_CSV_PATH,
        usecols=[CID_COL, NAME_COL],
        dtype='string',
        chunksize=2_000_000,
        on_bad_lines='skip',
    )
    for chunk in tqdm(reader, desc='Streaming reference CSV'):
        total_rows += len(chunk)
        chunk = chunk.dropna(subset=[CID_COL, NAME_COL]).copy()
        chunk['normalized_name'] = normalize_series(chunk[NAME_COL])
        chunk = chunk[chunk['normalized_name'].str.len() > 0]
        chunk = chunk.rename(columns={CID_COL: 'cid', NAME_COL: 'display_name'})
        chunk = chunk[['normalized_name', 'cid', 'display_name']].drop_duplicates(['normalized_name', 'cid'])
        if chunk.empty:
            continue
        con.register('chunk_df', chunk)
        con.execute('INSERT INTO reference_pairs SELECT normalized_name, CAST(cid AS VARCHAR), display_name FROM chunk_df')
        con.unregister('chunk_df')
        inserted_rows += len(chunk)

    con.execute('DROP TABLE IF EXISTS unique_surface_corpus')
    con.execute('''
        CREATE TABLE unique_surface_corpus AS
        WITH dedup AS (
            SELECT normalized_name, cid, MIN(display_name) AS display_name
            FROM reference_pairs
            GROUP BY normalized_name, cid
        ),
        grouped AS (
            SELECT
                normalized_name,
                ANY_VALUE(display_name) AS display_name,
                STRING_AGG(DISTINCT cid, '|') AS candidate_cids,
                COUNT(DISTINCT cid) AS n_candidate_cids
            FROM dedup
            GROUP BY normalized_name
        )
        SELECT
            ROW_NUMBER() OVER (ORDER BY normalized_name) - 1 AS surface_id,
            normalized_name,
            display_name,
            candidate_cids,
            n_candidate_cids
        FROM grouped
    ''')
    con.execute(f"COPY unique_surface_corpus TO '{q(CORPUS_PARQUET_PATH)}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    n_corpus = int(con.execute('SELECT COUNT(*) FROM unique_surface_corpus').fetchone()[0])
    con.close()

    summary = {
        'loaded_from_cache': False,
        'reference_rows_read': int(total_rows),
        'inserted_reference_pairs_after_chunk_dedup': int(inserted_rows),
        'unique_surface_rows': int(n_corpus),
        'elapsed_sec': float(time.perf_counter() - t0),
        'corpus_parquet_path': str(CORPUS_PARQUET_PATH),
    }
    save_json(summary, RESULTS_RUN_DIR / 'corpus_summary.json')
    backup_artifact(CORPUS_PARQUET_PATH, large=False)
    print(json.dumps(summary, indent=2))
    return summary


corpus_summary = prepare_unique_surface_corpus()


## V2 RefMet-to-CID Mapping


In [ ]:
def load_lab_query_pairs() -> pd.DataFrame:
    lab = pd.read_csv(LAB_CSV_PATH, dtype='string', engine='python', on_bad_lines='skip')
    missing = [col for col in [METABOLITE_COL, REFMET_COL] if col not in lab.columns]
    if missing:
        raise ValueError(f'Missing required lab columns: {missing}. Available columns: {list(lab.columns)}')
    lab = lab.reset_index(drop=True).copy()
    lab['source_row_id'] = np.arange(len(lab), dtype=np.int64)
    lab['metabolite_name'] = lab[METABOLITE_COL].astype('string').str.strip()
    lab['refmet_name'] = lab[REFMET_COL].astype('string').str.strip()
    lab['metabolite_normalized'] = normalize_series(lab['metabolite_name'])
    lab['refmet_normalized'] = normalize_series(lab['refmet_name'])
    has_metabolite = lab['metabolite_normalized'].str.len() > 0
    has_refmet = lab['refmet_normalized'].str.len() > 0
    out = lab[has_metabolite & has_refmet].copy()
    out = out.drop_duplicates(['metabolite_normalized', 'refmet_normalized']).reset_index(drop=True)
    out['query_i'] = np.arange(len(out), dtype=np.int64)
    return out


def aggregate_matches(matches: pd.DataFrame) -> pd.DataFrame:
    if matches.empty:
        return pd.DataFrame(columns=[
            'query_i', 'refmet_mapping_method', 'mapped_refmet_names',
            'mapped_refmet_display_names', 'refmet_candidate_cids', 'refmet_n_candidate_cids',
        ])
    priority = {
        'exact': 0,
        'strip_instrument_suffix': 1,
        'punctuation_spaced': 2,
        'remove_quotes': 3,
        'remove_terminal_count': 4,
        'compact_key': 5,
        'embedding_high_conf': 6,
    }
    matches = matches.copy()
    matches['priority'] = matches['method'].map(priority).fillna(99).astype(int)
    min_priority = matches.groupby('query_i')['priority'].transform('min')
    best = matches[matches['priority'].eq(min_priority)].copy()
    rows = []
    for query_i, group in best.groupby('query_i', sort=False):
        cids = pipe_join(group['candidate_cids'])
        names = '|'.join(pd.unique(group['mapped_name'].astype(str))[:20])
        display_names = '|'.join(pd.unique(group['display_name'].astype(str))[:20])
        methods = '|'.join(pd.unique(group['method'].astype(str)))
        rows.append({
            'query_i': int(query_i),
            'refmet_mapping_method': methods,
            'mapped_refmet_names': names,
            'mapped_refmet_display_names': display_names,
            'refmet_candidate_cids': cids,
            'refmet_n_candidate_cids': len(pipe_split(cids)),
        })
    return pd.DataFrame(rows)


def deterministic_refmet_mapping(candidate: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    variant_rows = []
    for row in candidate[['query_i', 'refmet_name']].itertuples(index=False):
        seen = set()
        for method, variant in make_refmet_variants(row.refmet_name):
            if variant and variant not in seen:
                seen.add(variant)
                variant_rows.append({'query_i': int(row.query_i), 'method': method, 'variant_name': variant})
    variants = pd.DataFrame(variant_rows)
    variants = variants.drop_duplicates(['query_i', 'method', 'variant_name'])

    con = duckdb.connect()
    con.register('variants', variants)
    variant_matches = con.execute(f'''
        SELECT
            v.query_i,
            v.method,
            u.normalized_name AS mapped_name,
            u.display_name,
            u.candidate_cids
        FROM variants v
        JOIN read_parquet('{q(CORPUS_PARQUET_PATH)}') u
            ON v.variant_name = u.normalized_name
    ''').df()

    all_matches = [variant_matches]
    mapped_query_ids = set(variant_matches['query_i'].astype(int).tolist()) if not variant_matches.empty else set()

    compact_matches = pd.DataFrame()
    if RUN_COMPACT_REFMET_MAPPING:
        remaining = candidate[~candidate['query_i'].isin(mapped_query_ids)].copy()
        compact = remaining[['query_i', 'refmet_normalized']].copy()
        compact['compact_refmet_key'] = compact['refmet_normalized'].map(compact_key)
        compact = compact[compact['compact_refmet_key'].str.len() >= int(MIN_COMPACT_KEY_LEN)]
        compact = compact.drop_duplicates(['query_i', 'compact_refmet_key'])
        con.register('compact_refmets', compact[['query_i', 'compact_refmet_key']])
        compact_matches = con.execute(f'''
            WITH raw AS (
                SELECT
                    c.query_i,
                    'compact_key' AS method,
                    u.normalized_name AS mapped_name,
                    u.display_name,
                    u.candidate_cids
                FROM read_parquet('{q(CORPUS_PARQUET_PATH)}') u
                JOIN compact_refmets c
                    ON regexp_replace(u.normalized_name, '[^a-z0-9]+', '', 'g') = c.compact_refmet_key
            ),
            counted AS (
                SELECT *, COUNT(*) OVER (PARTITION BY query_i) AS n_surface_matches
                FROM raw
            )
            SELECT query_i, method, mapped_name, display_name, candidate_cids
            FROM counted
            WHERE n_surface_matches <= {int(MAX_COMPACT_MATCHES_PER_REFMET)}
        ''').df()
        all_matches.append(compact_matches)
    con.close()

    matches = pd.concat(all_matches, ignore_index=True) if all_matches else pd.DataFrame()
    mapping = aggregate_matches(matches)
    out = candidate.merge(mapping, on='query_i', how='left')
    out['refmet_mapped_to_reference'] = out['refmet_candidate_cids'].fillna('').astype(str).str.len() > 0
    out['metabolite_equals_refmet_normalized'] = out['metabolite_normalized'].eq(out['refmet_normalized'])

    summary = {
        'lab_rows_with_both_names_deduped': int(len(candidate)),
        'mapped_queries_total': int(out['refmet_mapped_to_reference'].sum()),
        'unmapped_queries_total': int((~out['refmet_mapped_to_reference']).sum()),
        'mapped_by_method': out.loc[out['refmet_mapped_to_reference'], 'refmet_mapping_method'].value_counts(dropna=False).to_dict(),
        'compact_mapping_enabled': bool(RUN_COMPACT_REFMET_MAPPING),
        'queries_where_metabolite_equals_refmet': int(out['metabolite_equals_refmet_normalized'].sum()),
    }
    return out, summary


lab_pairs = load_lab_query_pairs()
all_v2_queries, mapping_summary = deterministic_refmet_mapping(lab_pairs)
eval_queries = all_v2_queries[all_v2_queries['refmet_mapped_to_reference']].reset_index(drop=True).copy()
eval_queries['eval_query_i'] = np.arange(len(eval_queries), dtype=np.int64)
all_v2_queries.to_csv(RESULTS_ARTIFACTS_DIR / 'v2_all_query_pairs_with_refmet_mapping.csv', index=False)
eval_queries.to_csv(RESULTS_ARTIFACTS_DIR / 'v2_eval_queries.csv', index=False)
save_json(mapping_summary, RESULTS_RUN_DIR / 'v2_mapping_summary.json')
print(json.dumps(mapping_summary, indent=2))
display(eval_queries.head(10))


## Encode or Restore Corpus and Query Embeddings


In [ ]:
import torch


def encode_parquet_texts_to_memmap(
    parquet_path: Path,
    text_col: str,
    out_path: Path,
    tokenizer,
    model,
    model_name: str,
    batch_size: int,
    device: str,
    run_key: str,
) -> np.ndarray:
    pf = pq.ParquetFile(parquet_path)
    n = int(pf.metadata.num_rows)
    d = int(model.config.hidden_size)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path = out_path.with_suffix(out_path.suffix + '.meta.json')
    expected_meta = {
        'n': n,
        'd': d,
        'text_col': text_col,
        'model_name': model_name,
        'pooling': POOLING,
        'max_length': int(MAX_LENGTH),
        'dtype': EMBED_DTYPE,
        'normalize': bool(L2_NORMALIZE),
        'run_key': run_key,
    }
    if out_path.exists() and meta_path.exists() and not REENCODE_CORPUS:
        old_meta = load_json_if_exists(meta_path)
        arr = np.load(out_path, mmap_mode='r')
        if old_meta == expected_meta and arr.shape == (n, d):
            print('Using cached corpus embeddings:', out_path)
            return arr
    if out_path.exists():
        out_path.unlink()
    arr = np.lib.format.open_memmap(
        out_path,
        mode='w+',
        dtype=np.float16 if EMBED_DTYPE == 'float16' else np.float32,
        shape=(n, d),
    )
    autocast_enabled = device.startswith('cuda') and torch.cuda.is_available()
    offset = 0
    t0 = time.perf_counter()
    with torch.inference_mode():
        for record_batch in tqdm(
            pf.iter_batches(batch_size=int(batch_size), columns=[text_col]),
            total=(n + int(batch_size) - 1) // int(batch_size),
            desc=f'Encoding {text_col}',
        ):
            texts = [str(value) for value in record_batch.column(0).to_pylist()]
            enc = tokenizer(texts, padding=True, truncation=True, max_length=int(MAX_LENGTH), return_tensors='pt').to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
                out = model(**enc)
                emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
                if L2_NORMALIZE:
                    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            end = offset + len(texts)
            arr[offset:end] = emb.detach().cpu().numpy().astype(arr.dtype, copy=False)
            offset = end
            del texts, enc, out, emb
            if device.startswith('cuda'):
                torch.cuda.empty_cache()
    arr.flush()
    save_json(expected_meta, meta_path)
    save_json({'elapsed_sec': float(time.perf_counter() - t0)}, out_path.with_suffix(out_path.suffix + '.timing.json'))
    return np.load(out_path, mmap_mode='r')


def encode_texts_to_memmap(
    texts: list[str],
    out_path: Path,
    tokenizer,
    model,
    model_name: str,
    batch_size: int,
    device: str,
    run_key: str,
    force_reencode: bool = False,
) -> np.ndarray:
    n = len(texts)
    d = int(model.config.hidden_size)
    meta_path = out_path.with_suffix(out_path.suffix + '.meta.json')
    expected_meta = {
        'n': n,
        'd': d,
        'model_name': model_name,
        'pooling': POOLING,
        'max_length': int(MAX_LENGTH),
        'dtype': EMBED_DTYPE,
        'normalize': bool(L2_NORMALIZE),
        'run_key': run_key,
    }
    if out_path.exists() and meta_path.exists() and not force_reencode:
        old_meta = load_json_if_exists(meta_path)
        arr = np.load(out_path, mmap_mode='r')
        if old_meta == expected_meta and arr.shape == (n, d):
            print('Using cached embeddings:', out_path)
            return arr
    if out_path.exists():
        out_path.unlink()
    arr = np.lib.format.open_memmap(
        out_path,
        mode='w+',
        dtype=np.float16 if EMBED_DTYPE == 'float16' else np.float32,
        shape=(n, d),
    )
    autocast_enabled = device.startswith('cuda') and torch.cuda.is_available()
    t0 = time.perf_counter()
    with torch.inference_mode():
        for start in tqdm(range(0, n, int(batch_size)), desc=f'Encoding {out_path.name}'):
            end = min(start + int(batch_size), n)
            batch = [str(x) for x in texts[start:end]]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=int(MAX_LENGTH), return_tensors='pt').to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
                out = model(**enc)
                emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
                if L2_NORMALIZE:
                    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            arr[start:end] = emb.detach().cpu().numpy().astype(arr.dtype, copy=False)
            del batch, enc, out, emb
            if device.startswith('cuda'):
                torch.cuda.empty_cache()
    arr.flush()
    save_json(expected_meta, meta_path)
    save_json({'elapsed_sec': float(time.perf_counter() - t0)}, out_path.with_suffix(out_path.suffix + '.timing.json'))
    return np.load(out_path, mmap_mode='r')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Embedding device:', device)
tokenizer, hf_model = load_hf_model(str(MODEL_DIR), device)

restore_artifact(CORPUS_EMBEDDINGS_PATH)
corpus_embeddings = encode_parquet_texts_to_memmap(
    CORPUS_PARQUET_PATH,
    text_col='normalized_name',
    out_path=CORPUS_EMBEDDINGS_PATH,
    tokenizer=tokenizer,
    model=hf_model,
    model_name=str(MODEL_DIR),
    batch_size=CORPUS_ENCODE_BATCH_SIZE,
    device=device,
    run_key='external_lab_unique_surface_sapbert_contrastive',
)

query_embeddings = encode_texts_to_memmap(
    eval_queries['metabolite_normalized'].astype(str).tolist(),
    QUERY_EMBEDDINGS_PATH,
    tokenizer,
    hf_model,
    str(MODEL_DIR),
    QUERY_ENCODE_BATCH_SIZE,
    device,
    run_key=RUN_NAME + '_queries',
    force_reencode=REENCODE_QUERIES,
)
print('Corpus embeddings:', corpus_embeddings.shape, corpus_embeddings.dtype)
print('Query embeddings:', query_embeddings.shape, query_embeddings.dtype)
backup_artifact(QUERY_EMBEDDINGS_PATH, large=False)


## Build or Load Main FAISS Index


In [ ]:
import faiss

if device.startswith('cuda'):
    print('Releasing transformer model GPU memory before FAISS...')
    del hf_model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


def expected_faiss_meta(n: int, d: int) -> dict:
    nlist_eff = min(max(int(NLIST), 1), max(int(n), 1), max(int(TRAIN_SIZE), 1)) if INDEX_TYPE != 'flat' else int(NLIST)
    return {
        'index_type': INDEX_TYPE,
        'n': int(n),
        'd': int(d),
        'nlist': int(nlist_eff),
        'train_size': int(TRAIN_SIZE),
        'm_pq': int(M_PQ),
        'nbits': int(NBITS),
        'seed': int(SEED),
        'run_key': 'external_lab_unique_surface_sapbert_contrastive',
    }


def move_faiss_index_to_device(cpu_index, faiss_device: str):
    if faiss_gpu_available(faiss, faiss_device):
        gpu_id = int(faiss_device.split(':', 1)[1]) if ':' in faiss_device else 0
        res = faiss.StandardGpuResources()
        opts = faiss.GpuClonerOptions()
        opts.useFloat16 = bool(USE_FLOAT16_FAISS)
        index = faiss.index_cpu_to_gpu(res, gpu_id, cpu_index, opts)
    else:
        index = cpu_index
    if hasattr(index, 'nprobe'):
        index.nprobe = int(NPROBE)
    return index


def load_cached_faiss_index(expected_meta: dict, faiss_device: str):
    restore_artifact(FAISS_INDEX_PATH)
    old_meta = load_json_if_exists(FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.meta.json'))
    if not FAISS_INDEX_PATH.exists() or old_meta != expected_meta:
        return None
    print('Using cached FAISS index:', FAISS_INDEX_PATH)
    return move_faiss_index_to_device(faiss.read_index(str(FAISS_INDEX_PATH)), faiss_device)


n_corpus, dim = corpus_embeddings.shape
ivfpq_codes_gb = n_corpus * M_PQ / (1024 ** 3)
flat_gpu_gb = n_corpus * dim * (2 if USE_FLOAT16_FAISS else 4) / (1024 ** 3)
estimated_index_gb = ivfpq_codes_gb if INDEX_TYPE == 'ivfpq' else flat_gpu_gb
faiss_device = 'cuda' if USE_GPU_FAISS_IF_FITS and estimated_index_gb <= GPU_INDEX_MEMORY_LIMIT_GB else 'cpu'
print('Estimated FAISS index GB:', round(estimated_index_gb, 2))
print('FAISS device:', faiss_device)

if REBUILD_FAISS_INDEX:
    for stale in [FAISS_INDEX_PATH, FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.meta.json')]:
        if stale.exists():
            stale.unlink()

faiss_expected_meta = expected_faiss_meta(n_corpus, dim)
faiss_index = None if REBUILD_FAISS_INDEX else load_cached_faiss_index(faiss_expected_meta, faiss_device)
if faiss_index is None:
    faiss_index, _ = build_faiss_index_from_memmap(
        xb_memmap=corpus_embeddings,
        index_path=FAISS_INDEX_PATH,
        index_type=INDEX_TYPE,
        nlist=NLIST,
        nprobe=NPROBE,
        train_size=TRAIN_SIZE,
        add_batch_size=ADD_BATCH_SIZE,
        m_pq=M_PQ,
        nbits=NBITS,
        seed=SEED,
        device=faiss_device,
        use_float16=USE_FLOAT16_FAISS,
        run_key='external_lab_unique_surface_sapbert_contrastive',
    )
    backup_artifact(FAISS_INDEX_PATH, large=True)


## Main SapBERT Candidate Search


In [ ]:
def search_queries(index, embeddings: np.ndarray, k_retrieve: int, batch_size: int, idx_path: Path, score_path: Path) -> tuple[np.ndarray, np.ndarray]:
    n = len(embeddings)
    idx = np.lib.format.open_memmap(idx_path, mode='w+', dtype=np.int64, shape=(n, int(k_retrieve)))
    scores = np.lib.format.open_memmap(score_path, mode='w+', dtype=np.float32, shape=(n, int(k_retrieve)))
    for start in tqdm(range(0, n, int(batch_size)), desc='FAISS search metabolite_name queries'):
        end = min(start + int(batch_size), n)
        xq = np.ascontiguousarray(np.asarray(embeddings[start:end], dtype=np.float32))
        d, i = index.search(xq, int(k_retrieve))
        idx[start:end] = i
        scores[start:end] = d
        del xq, d, i
    idx.flush()
    scores.flush()
    return np.load(idx_path, mmap_mode='r'), np.load(score_path, mmap_mode='r')


expected_search_meta = {
    'n_queries': int(len(eval_queries)),
    'k_retrieve': int(K_RETRIEVE),
    'query_embeddings_shape': tuple(query_embeddings.shape),
    'faiss_index_size': int(FAISS_INDEX_PATH.stat().st_size) if FAISS_INDEX_PATH.exists() else None,
    'run_name': RUN_NAME,
}
search_meta_path = NEIGHBOR_INDICES_PATH.with_suffix(NEIGHBOR_INDICES_PATH.suffix + '.meta.json')
old_search_meta = load_json_if_exists(search_meta_path)
if (
    NEIGHBOR_INDICES_PATH.exists()
    and NEIGHBOR_SCORES_PATH.exists()
    and old_search_meta == expected_search_meta
    and not RERUN_MAIN_SEARCH
):
    neighbor_indices = np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r')
    neighbor_scores = np.load(NEIGHBOR_SCORES_PATH, mmap_mode='r')
    print('Using cached main neighbors:', neighbor_indices.shape)
else:
    neighbor_indices, neighbor_scores = search_queries(
        faiss_index,
        query_embeddings,
        K_RETRIEVE,
        SEARCH_BATCH_SIZE,
        NEIGHBOR_INDICES_PATH,
        NEIGHBOR_SCORES_PATH,
    )
    save_json(expected_search_meta, search_meta_path)
print('Main neighbors:', neighbor_indices.shape)


## Optional RefMet Target Dictionary Search


In [ ]:
def prepare_refmet_target_dictionary(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    grouped = eval_df.groupby('refmet_normalized', sort=False)
    for aug_id, (refmet_norm, group) in enumerate(grouped):
        cids = pipe_join(group['refmet_candidate_cids'])
        rows.append({
            'aug_id': int(aug_id),
            'normalized_name': refmet_norm,
            'display_name': str(group['refmet_name'].iloc[0]),
            'candidate_cids': cids,
            'n_candidate_cids': len(pipe_split(cids)),
            'source': 'mapped_refmet_target_dictionary',
        })
    out = pd.DataFrame(rows)
    out.to_csv(RESULTS_ARTIFACTS_DIR / 'v2_refmet_target_dictionary.csv', index=False)
    return out


refmet_targets = prepare_refmet_target_dictionary(eval_queries)
print('RefMet target dictionary rows:', len(refmet_targets))

if EVALUATE_REFMET_TARGET_DICTIONARY:
    # Reload the model only for the small RefMet target dictionary embeddings.
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer, hf_model = load_hf_model(str(MODEL_DIR), device)
    refmet_target_embeddings = encode_texts_to_memmap(
        refmet_targets['normalized_name'].astype(str).tolist(),
        REFMET_TARGET_EMBEDDINGS_PATH,
        tokenizer,
        hf_model,
        str(MODEL_DIR),
        QUERY_ENCODE_BATCH_SIZE,
        device,
        run_key=RUN_NAME + '_refmet_target_dictionary',
        force_reencode=REENCODE_REFMET_TARGETS,
    )
    del hf_model
    del tokenizer
    gc.collect()
    if device.startswith('cuda'):
        torch.cuda.empty_cache()

    target_index = faiss.IndexFlatIP(refmet_target_embeddings.shape[1])
    target_index.add(np.ascontiguousarray(np.asarray(refmet_target_embeddings, dtype=np.float32)))
    target_search_meta = {
        'n_queries': int(len(eval_queries)),
        'n_refmet_targets': int(len(refmet_targets)),
        'k': int(REFMET_TARGET_K),
        'query_embeddings_shape': tuple(query_embeddings.shape),
    }
    target_search_meta_path = REFMET_TARGET_INDICES_PATH.with_suffix(REFMET_TARGET_INDICES_PATH.suffix + '.meta.json')
    if (
        REFMET_TARGET_INDICES_PATH.exists()
        and REFMET_TARGET_SCORES_PATH.exists()
        and load_json_if_exists(target_search_meta_path) == target_search_meta
        and not RERUN_REFMET_TARGET_SEARCH
    ):
        refmet_neighbor_indices = np.load(REFMET_TARGET_INDICES_PATH, mmap_mode='r')
        refmet_neighbor_scores = np.load(REFMET_TARGET_SCORES_PATH, mmap_mode='r')
        print('Using cached RefMet target neighbors:', refmet_neighbor_indices.shape)
    else:
        refmet_neighbor_indices, refmet_neighbor_scores = search_queries(
            target_index,
            query_embeddings,
            REFMET_TARGET_K,
            SEARCH_BATCH_SIZE,
            REFMET_TARGET_INDICES_PATH,
            REFMET_TARGET_SCORES_PATH,
        )
        save_json(target_search_meta, target_search_meta_path)
    print('RefMet target neighbors:', refmet_neighbor_indices.shape)
else:
    refmet_target_embeddings = None
    refmet_neighbor_indices = None
    refmet_neighbor_scores = None


## Reranking and Evaluation Helpers


In [ ]:
def candidate_rows_for_surface_ids(surface_ids: np.ndarray) -> pd.DataFrame:
    ids = pd.DataFrame({'surface_id': np.unique(surface_ids[surface_ids >= 0]).astype(np.int64)})
    if ids.empty:
        return pd.DataFrame(columns=['surface_id', 'candidate_normalized_name', 'candidate_name', 'candidate_cids', 'n_candidate_cids'])
    con = duckdb.connect()
    con.register('ids', ids)
    out = con.execute(f'''
        SELECT
            ids.surface_id,
            u.normalized_name AS candidate_normalized_name,
            u.display_name AS candidate_name,
            u.candidate_cids,
            u.n_candidate_cids
        FROM ids
        LEFT JOIN read_parquet('{q(CORPUS_PARQUET_PATH)}') u
            ON ids.surface_id = u.surface_id
    ''').df()
    con.close()
    return out


def metrics_from_ranks(rank_arr: np.ndarray, prefix: str = '') -> dict:
    finite = np.isfinite(rank_arr)
    n = len(rank_arr)
    out = {f'{prefix}MRR': float(np.sum(np.where(finite, 1.0 / rank_arr, 0.0)) / max(1, n))}
    for k in [1, 5, 10, 100, 256]:
        out[f'{prefix}HitRate@{int(k)}'] = float(np.mean(rank_arr <= int(k))) if n else 0.0
    return out


def exact_dot_scores(query_batch: np.ndarray, candidate_ids: np.ndarray, embedding_matrix: np.ndarray, device: str) -> np.ndarray:
    b, k = candidate_ids.shape
    d = query_batch.shape[1]
    flat_ids = candidate_ids.reshape(-1)
    flat_ids = np.maximum(flat_ids, 0)
    cand = np.asarray(embedding_matrix[flat_ids], dtype=np.float32).reshape(b, k, d)
    q_np = np.asarray(query_batch, dtype=np.float32)
    if device.startswith('cuda') and torch.cuda.is_available():
        with torch.inference_mode():
            q_t = torch.from_numpy(q_np).to(device)
            c_t = torch.from_numpy(cand).to(device)
            scores = torch.einsum('bkd,bd->bk', c_t, q_t).detach().cpu().numpy()
        del q_t, c_t
        torch.cuda.empty_cache()
        return scores.astype(np.float32)
    return np.einsum('bkd,bd->bk', cand, q_np).astype(np.float32)


def hybrid_score(row: dict) -> float:
    exact_norm = (float(row.get('exact_score', 0.0)) + 1.0) / 2.0
    return (
        0.72 * exact_norm
        + 0.08 * float(row.get('faiss_score_norm', 0.0))
        + 0.07 * float(row.get('sequence_ratio', 0.0))
        + 0.05 * float(row.get('char3_jaccard', 0.0))
        + 0.03 * float(row.get('token_jaccard', 0.0))
        + 0.05 * float(row.get('lipid_signature_match', 0.0))
        + 0.02 * float(row.get('lipid_class_match', 0.0))
        + 0.04 * float(row.get('query_exact_candidate', 0.0))
        - 0.04 * float(row.get('stereo_mismatch', 0.0))
    )


def evaluate_candidate_pools(include_refmet_dictionary: bool = False, batch_queries: int = 1000):
    strategies = ['faiss_ivfpq', 'exact_embedding_rerank', 'hybrid_lexical_rerank']
    ranks = {name: np.full(len(eval_queries), np.inf, dtype=np.float64) for name in strategies}
    exact_ranks = {name: np.full(len(eval_queries), np.inf, dtype=np.float64) for name in strategies}
    cid_ranks = {name: np.full(len(eval_queries), np.inf, dtype=np.float64) for name in strategies}
    prediction_rows = []
    top_rows = []

    query_lookup = eval_queries.set_index('eval_query_i')
    pool_k = min(int(RERANK_POOL_K), int(K_RETRIEVE))
    aug_lookup = refmet_targets.set_index('aug_id').to_dict('index') if include_refmet_dictionary else {}

    for q_start in tqdm(range(0, len(eval_queries), int(batch_queries)), desc='Evaluating V2 rerankers'):
        q_end = min(q_start + int(batch_queries), len(eval_queries))
        batch_ids = np.arange(q_start, q_end, dtype=np.int64)
        main_ids = np.asarray(neighbor_indices[q_start:q_end, :pool_k])
        main_faiss = np.asarray(neighbor_scores[q_start:q_end, :pool_k], dtype=np.float32)
        q_batch = np.asarray(query_embeddings[q_start:q_end], dtype=np.float32)
        main_exact = exact_dot_scores(q_batch, main_ids, corpus_embeddings, device='cuda' if torch.cuda.is_available() else 'cpu')

        main_meta = candidate_rows_for_surface_ids(main_ids.reshape(-1))
        main_map = main_meta.set_index('surface_id').to_dict('index')

        for local_i, eval_i in enumerate(batch_ids):
            qrow = query_lookup.loc[int(eval_i)]
            gold_norm = str(qrow['refmet_normalized'])
            gold_cids = qrow['refmet_candidate_cids']
            query_text = str(qrow['metabolite_normalized'])

            fs = main_faiss[local_i]
            fs_min = float(np.nanmin(fs))
            fs_max = float(np.nanmax(fs))
            fs_den = fs_max - fs_min if fs_max > fs_min else 1.0

            candidates = []
            seen_keys = set()
            for rank0, surface_id in enumerate(main_ids[local_i]):
                surface_id = int(surface_id)
                if surface_id < 0:
                    continue
                cand = main_map.get(surface_id, {})
                cand_norm = str(cand.get('candidate_normalized_name', ''))
                cand_cids = cand.get('candidate_cids', '')
                key = ('main', surface_id)
                if key in seen_keys:
                    continue
                seen_keys.add(key)
                features = lexical_features(query_text, cand_norm)
                row = {
                    'candidate_key': f'main:{surface_id}',
                    'candidate_source': 'pubchem_synonym',
                    'candidate_rank_faiss': int(rank0 + 1),
                    'surface_id': surface_id,
                    'candidate_name': cand.get('candidate_name', ''),
                    'candidate_normalized_name': cand_norm,
                    'candidate_cids': cand_cids,
                    'faiss_score': float(fs[rank0]),
                    'faiss_score_norm': float((fs[rank0] - fs_min) / fs_den),
                    'exact_score': float(main_exact[local_i, rank0]),
                    **features,
                }
                row['hybrid_score'] = hybrid_score(row)
                row['exact_refmet_match'] = bool(cand_norm == gold_norm)
                row['same_refmet_cid'] = has_cid_overlap(cand_cids, gold_cids)
                row['correct'] = bool(row['exact_refmet_match'] or row['same_refmet_cid'])
                candidates.append(row)

            if include_refmet_dictionary and refmet_neighbor_indices is not None:
                aug_ids = np.asarray(refmet_neighbor_indices[eval_i])
                aug_scores = np.asarray(refmet_neighbor_scores[eval_i], dtype=np.float32)
                aug_min = float(np.nanmin(aug_scores)) if len(aug_scores) else 0.0
                aug_max = float(np.nanmax(aug_scores)) if len(aug_scores) else 1.0
                aug_den = aug_max - aug_min if aug_max > aug_min else 1.0
                for rank0, aug_id in enumerate(aug_ids):
                    aug_id = int(aug_id)
                    cand = aug_lookup.get(aug_id, {})
                    cand_norm = str(cand.get('normalized_name', ''))
                    cand_cids = cand.get('candidate_cids', '')
                    key = ('refmet', aug_id)
                    if key in seen_keys:
                        continue
                    seen_keys.add(key)
                    features = lexical_features(query_text, cand_norm)
                    row = {
                        'candidate_key': f'refmet:{aug_id}',
                        'candidate_source': 'refmet_target_dictionary',
                        'candidate_rank_faiss': int(rank0 + 1),
                        'surface_id': -1 - aug_id,
                        'candidate_name': cand.get('display_name', ''),
                        'candidate_normalized_name': cand_norm,
                        'candidate_cids': cand_cids,
                        'faiss_score': float(aug_scores[rank0]),
                        'faiss_score_norm': float((aug_scores[rank0] - aug_min) / aug_den),
                        'exact_score': float(aug_scores[rank0]),
                        **features,
                    }
                    row['hybrid_score'] = hybrid_score(row)
                    row['exact_refmet_match'] = bool(cand_norm == gold_norm)
                    row['same_refmet_cid'] = has_cid_overlap(cand_cids, gold_cids)
                    row['correct'] = bool(row['exact_refmet_match'] or row['same_refmet_cid'])
                    candidates.append(row)

            ordered = {
                'faiss_ivfpq': sorted(candidates, key=lambda r: (r['candidate_source'] != 'pubchem_synonym', r['candidate_rank_faiss'])),
                'exact_embedding_rerank': sorted(candidates, key=lambda r: r['exact_score'], reverse=True),
                'hybrid_lexical_rerank': sorted(candidates, key=lambda r: r['hybrid_score'], reverse=True),
            }

            pred_payload = {
                'eval_query_i': int(eval_i),
                'source_row_id': int(qrow['source_row_id']),
                'metabolite_name': qrow['metabolite_name'],
                'refmet_name': qrow['refmet_name'],
                'metabolite_normalized': qrow['metabolite_normalized'],
                'refmet_normalized': qrow['refmet_normalized'],
                'refmet_candidate_cids': gold_cids,
                'refmet_mapping_method': qrow.get('refmet_mapping_method', ''),
            }
            for strategy, rows in ordered.items():
                best_correct = None
                first_exact_rank = None
                first_cid_rank = None
                for rank0, cand in enumerate(rows):
                    if cand['exact_refmet_match'] and first_exact_rank is None:
                        first_exact_rank = rank0 + 1
                        exact_ranks[strategy][eval_i] = rank0 + 1
                    if cand['same_refmet_cid'] and first_cid_rank is None:
                        first_cid_rank = rank0 + 1
                        cid_ranks[strategy][eval_i] = rank0 + 1
                    if cand['correct'] and best_correct is None:
                        best_correct = cand
                        ranks[strategy][eval_i] = rank0 + 1
                top1 = rows[0] if rows else {}
                pred_payload[f'{strategy}_top1_name'] = top1.get('candidate_name', '')
                pred_payload[f'{strategy}_top1_cids'] = top1.get('candidate_cids', '')
                pred_payload[f'{strategy}_top1_source'] = top1.get('candidate_source', '')
                pred_payload[f'{strategy}_top1_correct'] = bool(top1.get('correct', False))
                pred_payload[f'{strategy}_best_correct_rank'] = None if best_correct is None else int(ranks[strategy][eval_i])
                pred_payload[f'{strategy}_best_correct_name'] = '' if best_correct is None else best_correct.get('candidate_name', '')
                if strategy == 'hybrid_lexical_rerank':
                    for rank0, cand in enumerate(rows[:int(SAVE_TOP_K)]):
                        top_rows.append({
                            **pred_payload,
                            'rank': int(rank0 + 1),
                            **cand,
                        })
            prediction_rows.append(pred_payload)

    metrics = {}
    for strategy in strategies:
        metrics[strategy] = {
            **metrics_from_ranks(ranks[strategy]),
            **metrics_from_ranks(exact_ranks[strategy], prefix='ExactRefmet_'),
            **metrics_from_ranks(cid_ranks[strategy], prefix='SameCid_'),
        }
    return metrics, pd.DataFrame(prediction_rows), pd.DataFrame(top_rows)


## Run V2 Reranking Experiments


In [ ]:
main_metrics, main_predictions, main_top_hits = evaluate_candidate_pools(include_refmet_dictionary=False)
print(json.dumps(main_metrics, indent=2))

if EVALUATE_REFMET_TARGET_DICTIONARY:
    augmented_metrics, augmented_predictions, augmented_top_hits = evaluate_candidate_pools(include_refmet_dictionary=True)
    print(json.dumps(augmented_metrics, indent=2))
else:
    augmented_metrics, augmented_predictions, augmented_top_hits = {}, pd.DataFrame(), pd.DataFrame()


## Save Outputs


In [ ]:
comparison_rows = []
for candidate_set_name, metrics_by_strategy in [
    ('pubchem_only', main_metrics),
    ('pubchem_plus_refmet_target_dictionary', augmented_metrics),
]:
    for strategy, metric_values in metrics_by_strategy.items():
        row = {'candidate_set': candidate_set_name, 'strategy': strategy}
        row.update(metric_values)
        comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RESULTS_RUN_DIR / 'v2_corrected_reranker_comparison.csv', index=False)

main_predictions.to_csv(RESULTS_RUN_DIR / 'v2_corrected_predictions_pubchem_only.csv', index=False)
main_top_hits.to_csv(RESULTS_RUN_DIR / 'v2_corrected_top_hits_pubchem_only_hybrid.csv', index=False)
if not augmented_predictions.empty:
    augmented_predictions.to_csv(RESULTS_RUN_DIR / 'v2_corrected_predictions_augmented.csv', index=False)
    augmented_top_hits.to_csv(RESULTS_RUN_DIR / 'v2_corrected_top_hits_augmented_hybrid.csv', index=False)

all_metrics = {
    'run_name': RUN_NAME,
    'model_key': 'sapbert_contrastive',
    'task': 'v2_corrected_metabolite_to_refmet_or_same_cid',
    'n_lab_pairs_with_both_names_deduped': int(len(lab_pairs)),
    'n_evaluated_queries': int(len(eval_queries)),
    'mapping_summary': mapping_summary,
    'pubchem_only': main_metrics,
    'pubchem_plus_refmet_target_dictionary': augmented_metrics,
    'notes': [
        'faiss_ivfpq is the approximate candidate-generator order',
        'exact_embedding_rerank reorders candidates by exact dot product from cached embeddings',
        'hybrid_lexical_rerank adds lexical and chemistry-string features without using the gold target',
        'refmet target dictionary adds mapped RefMet labels as candidate surfaces searched with a small exact FAISS index',
    ],
}
save_json(all_metrics, RESULTS_RUN_DIR / 'v2_corrected_metrics.json')

artifact_manifest = pd.DataFrame([
    {'field': 'config', 'value': str(RESULTS_RUN_DIR / 'config.json')},
    {'field': 'metrics', 'value': str(RESULTS_RUN_DIR / 'v2_corrected_metrics.json')},
    {'field': 'mapping_summary', 'value': str(RESULTS_RUN_DIR / 'v2_mapping_summary.json')},
    {'field': 'comparison', 'value': str(RESULTS_RUN_DIR / 'v2_corrected_reranker_comparison.csv')},
    {'field': 'pubchem_predictions', 'value': str(RESULTS_RUN_DIR / 'v2_corrected_predictions_pubchem_only.csv')},
    {'field': 'pubchem_hybrid_top_hits', 'value': str(RESULTS_RUN_DIR / 'v2_corrected_top_hits_pubchem_only_hybrid.csv')},
    {'field': 'augmented_predictions', 'value': str(RESULTS_RUN_DIR / 'v2_corrected_predictions_augmented.csv')},
    {'field': 'augmented_hybrid_top_hits', 'value': str(RESULTS_RUN_DIR / 'v2_corrected_top_hits_augmented_hybrid.csv')},
])
artifact_manifest.to_csv(RESULTS_RUN_DIR / 'v2_corrected_artifact_manifest.csv', index=False)
display(comparison)
display(artifact_manifest)


## Plots and Decision Summary


In [ ]:
import matplotlib.pyplot as plt

plot_dir = RESULTS_RUN_DIR / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

if not comparison.empty:
    labels = comparison['candidate_set'] + '\n' + comparison['strategy']
    x = np.arange(len(comparison))
    fig, ax = plt.subplots(figsize=(max(10, len(comparison) * 1.3), 5))
    width = 0.22
    ax.bar(x - width, comparison['HitRate@1'], width, label='Hit@1')
    ax.bar(x, comparison['HitRate@10'], width, label='Hit@10')
    ax.bar(x + width, comparison['HitRate@256'], width, label='Hit@256')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Rate')
    ax.set_title('V2 corrected retrieval and reranking comparison')
    ax.legend()
    fig.tight_layout()
    fig.savefig(plot_dir / 'v2_hitrate_comparison.png', dpi=180, bbox_inches='tight')
    plt.show()

    fig, ax = plt.subplots(figsize=(max(10, len(comparison) * 1.3), 4))
    ax.bar(x, comparison['MRR'])
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('MRR')
    ax.set_title('V2 corrected MRR comparison')
    fig.tight_layout()
    fig.savefig(plot_dir / 'v2_mrr_comparison.png', dpi=180, bbox_inches='tight')
    plt.show()

baseline = comparison[(comparison['candidate_set'] == 'pubchem_only') & (comparison['strategy'] == 'faiss_ivfpq')]
best = comparison.sort_values('MRR', ascending=False).head(1)
print('Mapped/evaluable queries:', len(eval_queries), 'of', len(lab_pairs), 'deduplicated lab query pairs')
if not baseline.empty and not best.empty:
    b = baseline.iloc[0]
    best_row = best.iloc[0]
    print(f"Baseline FAISS MRR={b['MRR']:.4f}, Hit@10={b['HitRate@10']:.2%}, Hit@256={b['HitRate@256']:.2%}")
    print(f"Best V2 strategy: {best_row['candidate_set']} / {best_row['strategy']} MRR={best_row['MRR']:.4f}, Hit@1={best_row['HitRate@1']:.2%}, Hit@10={best_row['HitRate@10']:.2%}")
